# 🧩 머신러닝 확률론적 근거 — 보충 자료

이 노트북은 `ml_probability_foundations.ipynb`(본편)의 **보충 자료**이다.  
본편을 읽다가 "이건 왜 그렇지?" 하고 궁금해질 법한 부분들을 모아서 하나하나 깊이 파헤친다.

| 번호 | 보충 주제 | 본편 관련 섹션 |
|:---:|---------|:-----------:|
| 1 | 중심극한정리 직접 실험 — 요인 수를 바꿔보자! | 1.2 |
| 2 | MLE 수식에서 $2\sigma^2$는 어디로 갔나? | 1.3 → 1.4 |
| 3 | MAE의 확률론적 근거 — 라플라스 분포 | 1.5 |
| 4 | 승산비(Odds)의 직관적 이해 | 2.3 |
| 5 | 이진 교차 엔트로피(Binary Cross-Entropy) 도출 | 2장 전체 |
| 6 | 소프트맥스의 수치 안정성 트릭 | 3.2 코드 |
| 7 | 범주형 교차 엔트로피(Categorical Cross-Entropy) 도출 | 3장 전체 |
| 8 | 소프트맥스에 클래스 2개를 넣으면 시그모이드가 된다 | 3.2 |
| 9 | 연습 문제 / 자가 점검 퀴즈 | 전체 |

---
## 1. 중심극한정리 직접 실험 — 요인 수를 바꿔보자! 🔬

본편에서는 요인을 100개 합쳤을 때 정규분포가 나온다는 것을 보았다.  
하지만 진짜 "마법"을 느끼려면, **요인 수를 점점 늘려가면서** 종 모양이 서서히 나타나는 과정을 직접 눈으로 확인해 봐야 한다.

> 아래 셀의 `n_factors_list`를 자유롭게 수정해 보라!  
> 예를 들어 `[1, 2, 3, 5]`처럼 아주 작은 수를 넣으면 어떤 모양이 나올까?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(42)
n_samples = 10000

# ✏️ 직접 수정해 보자! 요인 수를 바꿔가며 분포 변화를 관찰하라.
n_factors_list = [1, 3, 10, 30, 100]

fig, axes = plt.subplots(1, len(n_factors_list), figsize=(4 * len(n_factors_list), 4), sharey=True)

for ax, n_factors in zip(axes, n_factors_list):
    factors = np.random.uniform(-1, 1, size=(n_samples, n_factors))
    errors = factors.sum(axis=1)

    x = np.linspace(errors.min(), errors.max(), 200)
    pdf = stats.norm.pdf(x, errors.mean(), errors.std())

    ax.hist(errors, bins=50, density=True, alpha=0.6, color='steelblue')
    ax.plot(x, pdf, 'r-', linewidth=2)
    ax.set_title(f'요인 {n_factors}개', fontsize=14, fontweight='bold')
    ax.set_xlabel('오차')
    if ax == axes[0]:
        ax.set_ylabel('밀도')
    ax.grid(True, alpha=0.3)

plt.suptitle('중심극한정리: 요인 수가 늘어날수록 정규분포(빨간 곡선)에 가까워진다!', fontsize=14, y=1.03)
plt.tight_layout()
plt.show()

print('👀 관찰 포인트:')
print('  - 요인 1개일 때: 균등분포의 네모난 직사각형 모양 그대로다.')
print('  - 요인 3개일 때: 살짝 종 모양이 보이기 시작한다.')
print('  - 요인 30개 이상: 빨간 정규분포 곡선과 거의 완벽하게 겹친다!')

### 💡 핵심 교훈

개별 요인이 아무리 기이한 분포(위 코드에서는 균등분포)를 따르더라도, **합쳐지는 요인 수가 충분히 많으면 그 합은 반드시 정규분포를 띤다.**  
실전에서는 요인이 30개 정도만 되어도 정규분포에 상당히 가까워지는 것을 확인할 수 있다.  
현실의 머신러닝 오차에는 수십, 수백 가지 미지의 원인이 뒤섞이므로 정규분포 가정은 매우 합리적이다.

---
## 2. MLE 수식에서 $2\sigma^2$는 어디로 갔나? 🔢

본편 섹션 1.3에서 정규분포의 확률밀도함수를 다음과 같이 보여주었다.

$$f(e_i) = \frac{1}{\sigma\sqrt{2\pi}} \exp\left(-\frac{e_i^2}{2\sigma^2}\right)$$

그런데 바로 다음 섹션 1.4에서 MLE를 전개할 때는 갑자기 $2\sigma^2$가 쏙 빠지고 아래처럼 단순해졌다.

$$P \propto \exp(-e_1^2) \times \exp(-e_2^2) \times \cdots$$

혹시 눈치챘는가? "$\frac{1}{2\sigma^2}$는 대체 어디 간 거지?" 하고 의아해할 수 있다.  
답은 아주 간단하다.

### 핵심: $\sigma$는 모든 샘플에 대해 동일한 '상수'이다

본편에서 우리는 **"모든 오차 $e_i$가 같은 정규분포 $\mathcal{N}(0, \sigma^2)$를 따른다"**고 가정했다 (등분산성).  
이 말은 $\sigma$의 값이 1번 샘플이든, 10000번 샘플이든 **전부 똑같은 고정된 상수**라는 뜻이다.

전체 우도(Likelihood)를 풀어 쓰면 이렇게 된다.

$$L = \prod_{i=1}^{n} \frac{1}{\sigma\sqrt{2\pi}} \exp\left(-\frac{e_i^2}{2\sigma^2}\right)$$

로그를 씌우면 (로그 우도):

$$\log L = -\frac{n}{2}\log(2\pi\sigma^2) - \frac{1}{2\sigma^2} \sum_{i=1}^{n} e_i^2$$

여기서 이 식을 **모델 파라미터(가중치 $w$)에 대해 최대화**한다고 해보자.  
그러면 $w$와 아무 상관없는 것들은 전부 상수 취급이 되어 최적화 결과에 영향을 주지 않는다.

- $-\frac{n}{2}\log(2\pi\sigma^2)$: $w$와 무관한 상수 → **무시!** ✂️  
- $-\frac{1}{2\sigma^2}$: $w$와 무관한 양의 상수 → 부호만 남기고 **무시!** ✂️

남는 것은 오직 이것뿐이다.

$$\arg\max_w \log L = \arg\min_w \sum_{i=1}^{n} e_i^2$$

> 따라서 $2\sigma^2$가 사라진 것이 아니라, **최적화 목표에 영향을 주지 않는 상수이기 때문에 생략한 것**이다.  
> 어떤 $\sigma$ 값이든, 오차 제곱합 $\sum e_i^2$를 최소화하는 가중치 $w$는 항상 같다!

In [ ]:
import numpy as np

# sigma 값이 달라져도 최소값을 만드는 지점(argmin)은 동일하다는 것을 확인하자.
errors = np.array([2.0, -1.5, 3.0, -0.5, 1.0])  # 고정된 오차 벡터
sum_sq = np.sum(errors**2)

print('오차 제곱합 (모든 sigma에 대해 동일):', sum_sq)
print()

for sigma in [0.5, 1.0, 2.0, 10.0]:
    log_likelihood = -len(errors)/2 * np.log(2 * np.pi * sigma**2) - sum_sq / (2 * sigma**2)
    print(f'  sigma = {sigma:5.1f}  →  로그 우도 = {log_likelihood:.4f}  (절대값은 달라도 argmin은 동일!)')

print()
print('💡 sigma가 바뀌면 로그 우도의 절대 크기는 변하지만,')
print('   "어떤 가중치 w가 이 값을 최대로 만드는가"의 답은 바뀌지 않는다!')

---
## 3. MAE의 확률론적 근거 — 라플라스 분포 📐

본편 섹션 1.5에서 "이상치가 많으면 RMSE 대신 MAE를 쓰라"고 했다.  
그런데 RMSE가 **정규분포** 가정에서 MLE로 필연적으로 도출되었듯,  
MAE도 어떤 확률분포 가정에서 필연적으로 도출된 것일까?

**그렇다!** MAE는 **라플라스 분포(Laplace Distribution)** 가정에서 도출된다.

### 정규분포 vs 라플라스 분포

| | 정규분포 (가우시안) | 라플라스 분포 |
|---|---|---|
| **모양** | 부드러운 종 모양 🔔 | 뾰족한 텐트/삼각형 모양 ⛺ |
| **꼬리** | 얇은 꼬리 (이상치에 민감) | 두꺼운 꼬리 (이상치에 둔감) |
| **확률밀도함수** | $\exp\left(-\frac{e^2}{2\sigma^2}\right)$ | $\exp\left(-\frac{|e|}{b}\right)$ |
| **지수 안의 핵심** | $e^{\mathbf{2}}$ (제곱) | $|e|$ (절대값) |
| **MLE → 손실함수** | $\sum e_i^2$ → **MSE** | $\sum |e_i|$ → **MAE** |

핵심을 보라! 정규분포의 지수 안에는 $e^2$(제곱)이 있어서 MSE가 나왔고,  
라플라스 분포의 지수 안에는 $|e|$(절대값)이 있어서 MAE가 나온다.  
**분포의 모양이 곧 손실 함수의 모양을 결정**하는 것이다!

라플라스 분포는 꼬리가 정규분포보다 두껍기 때문에, "큰 오차가 가끔 나타나도 별로 놀라운 일이 아니다"라고 판단한다.  
그래서 이상치에 덜 휘둘리게 되는 것이다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

x = np.linspace(-6, 6, 500)

# 정규분포와 라플라스 분포 비교 (둘 다 평균=0, 비슷한 스케일)
normal_pdf = stats.norm.pdf(x, 0, 1)
laplace_pdf = stats.laplace.pdf(x, 0, 1 / np.sqrt(2))  # 분산을 맞추기 위해 scale 조정

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 왼쪽: 전체 모양 비교
axes[0].plot(x, normal_pdf, 'b-', linewidth=2, label='정규분포 (→ MSE/RMSE)')
axes[0].plot(x, laplace_pdf, 'r-', linewidth=2, label='라플라스 분포 (→ MAE)')
axes[0].set_title('정규분포 vs 라플라스 분포', fontsize=14)
axes[0].set_xlabel('오차 (e)')
axes[0].set_ylabel('확률 밀도')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)
axes[0].annotate('가운데가 더 뾰족하다\n→ 작은 오차 집중', xy=(0, 0.7), xytext=(2, 0.6),
                 arrowprops=dict(arrowstyle='->', color='red'), color='red', fontsize=10)

# 오른쪽: 꼬리 부분 확대
axes[1].plot(x, normal_pdf, 'b-', linewidth=2, label='정규분포')
axes[1].plot(x, laplace_pdf, 'r-', linewidth=2, label='라플라스 분포')
axes[1].set_title('꼬리(이상치 영역) 확대', fontsize=14)
axes[1].set_xlabel('오차 (e)')
axes[1].set_ylabel('확률 밀도')
axes[1].set_ylim(0, 0.05)
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)
axes[1].annotate('라플라스의 꼬리가 더 두껍다\n→ 이상치를 "흔한 일"로 본다', xy=(3.5, 0.02), xytext=(1.5, 0.04),
                 arrowprops=dict(arrowstyle='->', color='red'), color='red', fontsize=10)

plt.tight_layout()
plt.show()

print('📌 정규분포: 큰 오차가 나오면 "말도 안 돼!" → 제곱으로 벌칙을 무겁게 때린다 (MSE)')
print('📌 라플라스: 큰 오차가 나와도 "그럴 수도 있지" → 절대값으로 가볍게 벌칙을 준다 (MAE)')

---
## 4. 승산비(Odds)의 직관적 이해 🎰

본편 섹션 2.3에서 로짓 변환 $\log\left(\frac{p}{1-p}\right)$이 등장했다.  
그런데 분수 안의 $\frac{p}{1-p}$가 정확히 무엇을 뜻하는지 살펴보지 않고 넘어갔다.  
이 분수에는 **승산비(Odds)**라는 이름이 붙어 있다.

### 승산비란? 🐎

경마를 생각해 보자. 어떤 말이 이길 확률이 $p = 0.75$ (75%)라면,

$$\text{승산비(Odds)} = \frac{p}{1-p} = \frac{0.75}{0.25} = 3$$

이 말은 **"이 말이 지는 1번 동안 이기는 횟수가 3번"**, 즉 **"3대 1로 이긴다"**라는 뜻이다.  
카지노나 도박장에서 흔히 쓰이는 배당률이 바로 이 승산비의 역수이다.

| 확률 $p$ | 승산비 $\frac{p}{1-p}$ | 해석 |
|---------|-------------------|------|
| 0.5 | 1.0 | 반반이다 (1대 1) |
| 0.8 | 4.0 | 4대 1로 이긴다 |
| 0.9 | 9.0 | 9대 1로 이긴다 |
| 0.99 | 99.0 | 99대 1로 이긴다 |
| 0.01 | 0.01 | 99대 1로 진다 |

### 왜 승산비에 로그를 씌우나?

승산비 자체는 0부터 $+\infty$ 사이의 값만 가질 수 있다.  
여기에 **로그(log)**를 씌우면 범위가 $-\infty$에서 $+\infty$로 확장된다.

- $p < 0.5$ → 승산비 < 1 → $\log$(승산비) < 0 (**음수**)
- $p = 0.5$ → 승산비 = 1 → $\log$(승산비) = 0 (**영**)
- $p > 0.5$ → 승산비 > 1 → $\log$(승산비) > 0 (**양수**)

이 변환 덕분에 0~1 사이에 갇혀 있던 확률 $p$가 선형 회귀의 출력값($-\infty$ ~ $+\infty$)과 자연스럽게 대응하게 된다.  
그리고 이 관계를 $p$에 대해 역으로 풀면 시그모이드 함수가 탄생하는 것이다!

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

p = np.linspace(0.01, 0.99, 200)
odds = p / (1 - p)
log_odds = np.log(odds)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 왼쪽: 확률 → 승산비
axes[0].plot(p, odds, 'b-', linewidth=2)
axes[0].axhline(y=1, color='gray', linestyle='--', alpha=0.5)
axes[0].set_title('확률 → 승산비 (Odds)', fontsize=14)
axes[0].set_xlabel('확률 p')
axes[0].set_ylabel('승산비 p/(1-p)')
axes[0].set_ylim(0, 20)
axes[0].grid(True, alpha=0.3)
axes[0].annotate('p=0.5일 때 승산비=1\n(반반)', xy=(0.5, 1), xytext=(0.2, 8),
                 arrowprops=dict(arrowstyle='->', color='red'), color='red', fontsize=11)

# 오른쪽: 확률 → 로짓 (로그 승산비)
axes[1].plot(p, log_odds, 'r-', linewidth=2)
axes[1].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
axes[1].set_title('확률 → 로짓 = log(Odds)', fontsize=14)
axes[1].set_xlabel('확률 p')
axes[1].set_ylabel('로짓 log(p/(1-p))')
axes[1].grid(True, alpha=0.3)
axes[1].annotate('p=0.5이면 로짓=0', xy=(0.5, 0), xytext=(0.15, 2),
                 arrowprops=dict(arrowstyle='->', color='blue'), color='blue', fontsize=11)

plt.tight_layout()
plt.show()

print('📌 왼쪽 그래프: 확률이 1에 가까워지면 승산비가 급격히 폭발한다 (비대칭).')
print('📌 오른쪽 그래프: 로그를 씌우니 -∞ ~ +∞의 대칭적인 S자 곡선이 되었다!')
print('   이 오른쪽 그래프를 x, y축을 바꿔서 보면? 바로 시그모이드 함수다!')

---
## 5. 이진 교차 엔트로피(Binary Cross-Entropy) 도출 ⚖️

본편에서는 "정규분포 가정 → MLE → **MSE/RMSE**" 도출을 아주 명쾌하게 보여주었다.  
그런데 로지스틱 회귀(시그모이드)에서는 함수만 도출하고, 그에 대응하는 **손실 함수**는 다루지 않았다.

사실 로지스틱 회귀의 손실 함수인 **이진 교차 엔트로피(Binary Cross-Entropy, BCE)**도  
RMSE와 완전히 똑같은 원리로, **베르누이 분포 + MLE**에서 도출된다!

### 도출 과정

**1단계**: 하나의 샘플 $i$에 대해, 실제 정답이 $y_i$ (0 또는 1)이고 모델의 예측 확률이 $\hat{p}_i$라 하자.  
베르누이 분포의 확률질량함수에 의해, 이 정답이 나올 확률은 다음과 같다.

$$P(y_i | \hat{p}_i) = \hat{p}_i^{y_i} \cdot (1 - \hat{p}_i)^{1 - y_i}$$

이 식이 어렵게 느껴지면 $y_i$에 0과 1을 직접 넣어보라.
- $y_i = 1$ (스팸)이면: $P = \hat{p}_i^1 \cdot (1-\hat{p}_i)^0 = \hat{p}_i$ → 모델이 예측한 확률 그 자체!
- $y_i = 0$ (정상)이면: $P = \hat{p}_i^0 \cdot (1-\hat{p}_i)^1 = 1 - \hat{p}_i$ → 정상일 확률!

**2단계**: 전체 $n$개 샘플이 독립이므로 전체 우도는 곱셈이다.

$$L = \prod_{i=1}^{n} \hat{p}_i^{y_i} \cdot (1 - \hat{p}_i)^{1-y_i}$$

**3단계**: 로그를 씌우면 곱셈이 덧셈으로 바뀐다.

$$\log L = \sum_{i=1}^{n} \left[ y_i \log \hat{p}_i + (1 - y_i) \log(1 - \hat{p}_i) \right]$$

**4단계**: MLE는 이것을 최대화한다. 머신러닝에서는 관례적으로 **최소화** 문제로 바꾸기 위해 부호를 뒤집는다.

$$\text{BCE} = -\frac{1}{n}\sum_{i=1}^{n} \left[ y_i \log \hat{p}_i + (1 - y_i) \log(1 - \hat{p}_i) \right]$$

이것이 바로 **이진 교차 엔트로피(Binary Cross-Entropy)** 손실 함수이다! 🎉

> **결론**: RMSE가 정규분포에서 나왔듯, BCE는 **베르누이 분포 + MLE**에서 수학적으로 필연적으로 도출된다.  
> 본편의 세 섹션이 이제 완벽한 대칭을 이룬다:
>
> | 확률 가정 | MLE → 활성화 함수 | MLE → 손실 함수 |
> |---------|----------------|---------------|
> | 정규분포 | (없음) | **MSE / RMSE** |
> | 베르누이 분포 | **시그모이드** | **이진 교차 엔트로피** |
> | 카테고리 분포 | **소프트맥스** | **범주형 교차 엔트로피** |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

p_hat = np.linspace(0.001, 0.999, 500)

# y=1일 때의 손실: -log(p_hat)
loss_y1 = -np.log(p_hat)
# y=0일 때의 손실: -log(1-p_hat)
loss_y0 = -np.log(1 - p_hat)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(p_hat, loss_y1, 'r-', linewidth=2)
axes[0].set_title('실제 정답이 스팸(y=1)일 때의 손실', fontsize=13)
axes[0].set_xlabel('모델이 예측한 스팸 확률 (p̂)')
axes[0].set_ylabel('손실 = -log(p̂)')
axes[0].grid(True, alpha=0.3)
axes[0].annotate('p̂≈1 (스팸을 스팸이라 했다)\n→ 손실 ≈ 0 ✅', xy=(0.9, 0.1), xytext=(0.3, 1.5),
                 arrowprops=dict(arrowstyle='->', color='green'), color='green', fontsize=11)
axes[0].annotate('p̂≈0 (스팸을 정상이라 했다)\n→ 손실 폭발! ❌', xy=(0.05, 3), xytext=(0.3, 3.5),
                 arrowprops=dict(arrowstyle='->', color='red'), color='red', fontsize=11)

axes[1].plot(p_hat, loss_y0, 'b-', linewidth=2)
axes[1].set_title('실제 정답이 정상(y=0)일 때의 손실', fontsize=13)
axes[1].set_xlabel('모델이 예측한 스팸 확률 (p̂)')
axes[1].set_ylabel('손실 = -log(1-p̂)')
axes[1].grid(True, alpha=0.3)
axes[1].annotate('p̂≈0 (정상을 정상이라 했다)\n→ 손실 ≈ 0 ✅', xy=(0.1, 0.1), xytext=(0.35, 1.5),
                 arrowprops=dict(arrowstyle='->', color='green'), color='green', fontsize=11)
axes[1].annotate('p̂≈1 (정상을 스팸이라 했다)\n→ 손실 폭발! ❌', xy=(0.95, 3), xytext=(0.35, 3.5),
                 arrowprops=dict(arrowstyle='->', color='red'), color='red', fontsize=11)

plt.tight_layout()
plt.show()

print('📌 모델이 정답을 맞히면 손실이 0에 가깝고, 완전히 틀리면 손실이 무한대로 폭발한다.')
print('   이 "틀리면 호되게 벌주는" 성질이 모델을 올바른 방향으로 학습시킨다!')

---
## 6. 소프트맥스의 수치 안정성 트릭 🛡️

본편 섹션 3.2의 코드에서 소프트맥스를 다음과 같이 구현했다.

```python
def softmax(z):
    exp_z = np.exp(z - np.max(z))  # ← 왜 최댓값을 빼는가?
    return exp_z / exp_z.sum()
```

수학적으로 소프트맥스의 정의는 $p_k = \frac{\exp(z_k)}{\sum_j \exp(z_j)}$이다.  
그런데 코드에서는 분자와 분모 모두에서 `np.max(z)`를 빼고 있다. 왜 그런가?

### 문제: `exp`의 오버플로우 폭발 💥

만약 모델 내부 점수 $z_k$가 매우 큰 값(예: 1000)이라면,  
$\exp(1000)$은 컴퓨터가 표현할 수 있는 범위를 초과하여 **무한대(`inf`)**가 되어 버린다.  
그 결과 `inf / inf = NaN`이 되어 계산이 완전히 망가진다.

### 해결: 최댓값을 빼도 결과는 동일하다! 🧠

$c = \max(z)$라고 두면, 소프트맥스 분자·분모 모두에 $\exp(-c)$를 곱하는 것과 같다.

$$p_k = \frac{\exp(z_k)}{\sum_j \exp(z_j)} = \frac{\exp(z_k) \cdot \exp(-c)}{\sum_j \exp(z_j) \cdot \exp(-c)} = \frac{\exp(z_k - c)}{\sum_j \exp(z_j - c)}$$

분자·분모에 같은 값을 곱한 것이므로 **수학적 결과는 100% 동일**하다.  
하지만 이제 지수 안의 최댓값이 0이 되므로 $\exp$ 값이 절대로 폭발하지 않는다!

In [ ]:
import numpy as np

# 위험한 버전 (오버플로우 발생)
def softmax_naive(z):
    exp_z = np.exp(z)
    return exp_z / exp_z.sum()

# 안전한 버전 (최댓값 빼기 트릭)
def softmax_stable(z):
    exp_z = np.exp(z - np.max(z))
    return exp_z / exp_z.sum()

# 작은 점수: 둘 다 정상 작동
z_small = np.array([1.0, 2.0, 3.0])
print('=== 작은 점수 [1, 2, 3] ===')
print(f'  순진한 버전: {softmax_naive(z_small)}')
print(f'  안전한 버전: {softmax_stable(z_small)}')
print(f'  결과가 동일한가? {np.allclose(softmax_naive(z_small), softmax_stable(z_small))}')
print()

# 큰 점수: 순진한 버전은 폭발!
z_big = np.array([1000.0, 1001.0, 1002.0])
print('=== 큰 점수 [1000, 1001, 1002] ===')

import warnings
with warnings.catch_warnings():
    warnings.simplefilter('ignore', RuntimeWarning)
    naive_result = softmax_naive(z_big)
    print(f'  순진한 버전: {naive_result}  ← 💥 NaN 발생! 계산 불가!')

stable_result = softmax_stable(z_big)
print(f'  안전한 버전: {stable_result}  ← ✅ 정상 작동!')
print()
print('💡 최댓값을 빼는 단 한 줄의 트릭이 오버플로우를 완벽하게 방지한다!')

---
## 7. 범주형 교차 엔트로피(Categorical Cross-Entropy) 도출 📊

섹션 5에서 베르누이 분포(이진 분류)로부터 **이진 교차 엔트로피**를 도출했다.  
이제 카테고리 분포(다중 분류)로부터 **범주형 교차 엔트로피**를 도출해 보자.  
논리 구조는 정확히 동일하다!

### 도출 과정

**1단계**: 클래스가 $K$개이고, 하나의 샘플 $i$의 실제 정답이 원-핫 벡터 $\mathbf{y}_i = (y_{i,1}, \dots, y_{i,K})$라 하자.  
(예: 숫자 5 → $[0, 0, 0, 0, 0, 1, 0, 0, 0, 0]$)

카테고리 분포에 의해, 이 정답이 나올 확률은:

$$P(\mathbf{y}_i | \hat{\mathbf{p}}_i) = \prod_{k=1}^{K} \hat{p}_{i,k}^{y_{i,k}}$$

원-핫 벡터이므로 정답 클래스의 $y_{i,k}$만 1이고 나머지는 0이다.  
따라서 이 식은 결국 **"모델이 정답 클래스에 부여한 확률"** 하나만 살아남게 된다.

**2단계**: 전체 $n$개 샘플의 우도:

$$L = \prod_{i=1}^{n} \prod_{k=1}^{K} \hat{p}_{i,k}^{y_{i,k}}$$

**3단계**: 로그를 씌우면:

$$\log L = \sum_{i=1}^{n} \sum_{k=1}^{K} y_{i,k} \log \hat{p}_{i,k}$$

**4단계**: 부호를 뒤집어 최소화 문제로 바꾸면:

$$\text{CCE} = -\frac{1}{n}\sum_{i=1}^{n} \sum_{k=1}^{K} y_{i,k} \log \hat{p}_{i,k}$$

이것이 **범주형 교차 엔트로피(Categorical Cross-Entropy)** 손실 함수이다!

> 💡 **특별한 관계**: 클래스가 딱 2개($K=2$)일 때 위 공식을 전개하면,  
> 섹션 5에서 도출한 **이진 교차 엔트로피**와 정확히 동일해진다.  
> 소프트맥스가 시그모이드의 일반화인 것처럼, CCE도 BCE의 일반화이다!

In [ ]:
import numpy as np

def softmax(z):
    exp_z = np.exp(z - np.max(z))
    return exp_z / exp_z.sum()

def categorical_cross_entropy(y_true, y_pred):
    """범주형 교차 엔트로피 계산 (단일 샘플)"""
    # 로그 0 방지를 위해 작은 값(epsilon) 추가
    epsilon = 1e-15
    y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
    return -np.sum(y_true * np.log(y_pred))

# 실제 정답: 숫자 5 (원-핫 벡터)
y_true = np.array([0, 0, 0, 0, 0, 1, 0, 0, 0, 0])

print('=== 범주형 교차 엔트로피: 모델의 확신도에 따른 손실 변화 ===')
print()

# 시나리오 1: 모델이 정답(5)에 높은 확률을 부여
scores_good = np.array([-2, -3, -1, -3, -1, 5, -2, -2, -2, -2], dtype=float)
probs_good = softmax(scores_good)
loss_good = categorical_cross_entropy(y_true, probs_good)
print(f'시나리오 1: 모델이 숫자 5에 확신 (prob={probs_good[5]:.4f})')
print(f'  → 손실 = {loss_good:.4f}  ← 낮다! 모델이 잘 맞혔다 ✅')
print()

# 시나리오 2: 모델이 헷갈려하는 경우
scores_confused = np.array([0, 0, 0, 0, 0, 1, 0, 0, 0, 0], dtype=float)
probs_confused = softmax(scores_confused)
loss_confused = categorical_cross_entropy(y_true, probs_confused)
print(f'시나리오 2: 모델이 조금만 확신 (prob={probs_confused[5]:.4f})')
print(f'  → 손실 = {loss_confused:.4f}  ← 중간 정도의 벌점')
print()

# 시나리오 3: 모델이 완전히 틀린 경우
scores_wrong = np.array([5, -2, -2, -2, -2, -3, -2, -2, -2, -2], dtype=float)
probs_wrong = softmax(scores_wrong)
loss_wrong = categorical_cross_entropy(y_true, probs_wrong)
print(f'시나리오 3: 모델이 숫자 0이라 착각 (5에 대한 prob={probs_wrong[5]:.4f})')
print(f'  → 손실 = {loss_wrong:.4f}  ← 높다! 모델이 크게 틀렸다 ❌')

---
## 8. 소프트맥스에 클래스 2개를 넣으면 시그모이드가 된다 👬

본편의 마지막 표에서 "로지스틱 회귀는 소프트맥스 회귀의 특수한 경우"라고 했다.  
이를 수식으로 직접 증명해 보자.

### 증명

클래스가 **딱 2개** (클래스 0, 클래스 1)이고 모델 점수가 $z_0$, $z_1$이라 하자.  
소프트맥스로 클래스 1의 확률을 구하면:

$$P(y=1) = \frac{\exp(z_1)}{\exp(z_0) + \exp(z_1)}$$

**분자·분모를 $\exp(z_1)$로 나누면:**

$$P(y=1) = \frac{1}{\exp(z_0 - z_1) + 1} = \frac{1}{1 + \exp(-(z_1 - z_0))}$$

여기서 $x = z_1 - z_0$ (두 클래스 점수의 차이)이라고 치환하면:

$$P(y=1) = \frac{1}{1 + \exp(-x)} = \sigma(x)$$

이것은 **완벽하게 시그모이드 함수**이다! 🎉

### 직관적 해석

- 다중 분류(N개)에서는 각 클래스의 **절대적인 점수**가 중요하다.
- 이진 분류(2개)에서는 두 선수 중 누가 이기느냐의 **상대적 점수 차이($z_1 - z_0$)** 하나만 알면 충분하다.
- 그래서 로지스틱 회귀에서는 내부적으로 점수를 하나만 출력하고, 그것을 시그모이드에 넣는 것이다!

In [ ]:
import numpy as np

def softmax(z):
    exp_z = np.exp(z - np.max(z))
    return exp_z / exp_z.sum()

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

print('=== 소프트맥스(클래스 2개) vs 시그모이드 비교 ===')
print()

test_cases = [
    (1.0, 3.0),
    (-2.0, 2.0),
    (0.0, 0.0),
    (5.0, -1.0),
    (-10.0, 10.0),
]

print(f'{"z0":>6s}  {"z1":>6s}  │  {"소프트맥스 P(y=1)":>18s}  {"시그모이드 σ(z1-z0)":>20s}  {"동일?"}') 
print('─' * 75)

for z0, z1 in test_cases:
    softmax_p1 = softmax(np.array([z0, z1]))[1]
    sigmoid_p1 = sigmoid(z1 - z0)
    match = '✅' if np.isclose(softmax_p1, sigmoid_p1) else '❌'
    print(f'{z0:6.1f}  {z1:6.1f}  │  {softmax_p1:18.10f}  {sigmoid_p1:20.10f}  {match}')

print()
print('💡 모든 경우에 소수점 10자리까지 완벽하게 일치한다!')
print('   소프트맥스에 클래스를 2개만 넣으면 시그모이드와 수학적으로 동일해진다는 증명 완료! 🎉')

---
## 9. 연습 문제 / 자가 점검 퀴즈 📝

본편과 보충 자료의 핵심 개념을 잘 이해했는지 스스로 점검해 보자!

### 퀴즈 1: 중심극한정리

다음 중 **옳은** 것을 모두 고르라.

- (a) 중심극한정리는 원래 분포가 정규분포일 때만 성립한다.
- (b) 독립적인 확률변수 100개의 합은 원래 분포와 상관없이 정규분포에 가까워진다.
- (c) 합쳐지는 변수가 3개뿐이어도 완벽한 정규분포가 된다.
- (d) 머신러닝에서 오차가 정규분포를 따른다고 가정하는 근거가 중심극한정리이다.

<details>
<summary>🔑 정답 보기 (클릭)</summary>

**정답: (b), (d)**

- (a) ❌ 원래 분포가 무엇이든(균등분포, 지수분포 등) 상관없이 성립한다.
- (b) ✅ 이것이 중심극한정리의 핵심이다.
- (c) ❌ 3개 정도로는 정규분포에 충분히 가깝지 않다. 보통 30개 이상이면 꽤 가까워진다.
- (d) ✅ 오차가 수많은 미지의 원인의 합이므로 정규분포를 따른다고 가정하는 것이 합리적이다.
</details>

### 퀴즈 2: RMSE vs MAE

오차가 $[1, -1, 2, -2, 50]$일 때, RMSE와 MAE를 직접 계산하라.  
아래 셀에 코드를 작성한 뒤, 이상치(50)가 두 지표에 미치는 영향의 차이를 한 문장으로 설명하라.

In [ ]:
import numpy as np

errors = np.array([1, -1, 2, -2, 50])

# ✏️ 직접 계산해 보자!
# rmse = ???
# mae = ???

# print(f'RMSE: {rmse:.2f}')
# print(f'MAE:  {mae:.2f}')

<details>
<summary>🔑 정답 보기 (클릭)</summary>

```python
rmse = np.sqrt(np.mean(errors**2))  # ≈ 22.41
mae = np.mean(np.abs(errors))       # = 11.2
```

RMSE(22.41)가 MAE(11.2)보다 약 2배 크다. 이상치 50이 제곱되어 2500이 되면서 RMSE를 크게 끌어올렸기 때문이다.  
이상치가 많은 데이터에서는 MAE가 더 안정적인 성능 지표이다.
</details>

### 퀴즈 3: 시그모이드와 로짓

다음 빈칸을 채워라.

1. 시그모이드 함수에 $x=0$을 넣으면 출력값은 \_\_\_이다.
2. 로짓 변환에서 $p=0.5$를 넣으면 로짓 값은 \_\_\_이다.
3. 위 1번과 2번의 결과가 같은 이유는 시그모이드와 로짓이 서로 \_\_\_ 관계이기 때문이다.

<details>
<summary>🔑 정답 보기 (클릭)</summary>

1. **0.5** ($\frac{1}{1+e^0} = \frac{1}{2}$)
2. **0** ($\log\frac{0.5}{0.5} = \log 1 = 0$)
3. **역함수(inverse function)** 관계이기 때문이다.  
   로짓이 확률 → 실수로 변환하고, 시그모이드가 실수 → 확률로 되돌린다.
</details>

### 퀴즈 4: 소프트맥스

모델이 3개 클래스에 대해 내부 점수 $z = [2, 1, 0]$을 출력했다.  
아래 셀에서 소프트맥스를 직접 계산해 보라. (수치 안정성 트릭도 적용할 것!)

In [ ]:
import numpy as np

z = np.array([2, 1, 0], dtype=float)

# ✏️ 직접 구현해 보자!
# 1단계: 수치 안정성을 위해 최댓값을 빼라
# z_stable = ???

# 2단계: exp를 씌워라
# exp_z = ???

# 3단계: 합으로 나눠라
# probs = ???

# print(f'소프트맥스 결과: {probs}')
# print(f'확률의 합: {probs.sum():.6f}')

<details>
<summary>🔑 정답 보기 (클릭)</summary>

```python
z_stable = z - np.max(z)         # [0, -1, -2]
exp_z = np.exp(z_stable)         # [1.0, 0.3679, 0.1353]
probs = exp_z / exp_z.sum()      # [0.6652, 0.2447, 0.0900]
```

확률의 합은 정확히 1.0이 된다. 점수가 가장 높은 클래스 0이 약 66.5%의 확률을 차지한다.
</details>

### 퀴즈 5: 전체 통합 문제

다음 표의 빈칸을 채워라.

| 확률 가정 | MLE로 도출되는 활성화 함수 | MLE로 도출되는 손실 함수 |
|---------|:-------------------:|:-------------------:|
| 정규분포 | (해당 없음) | (1) \_\_\_ |
| 베르누이 분포 | (2) \_\_\_ | (3) \_\_\_ |
| 카테고리 분포 | (4) \_\_\_ | (5) \_\_\_ |
| 라플라스 분포 | (해당 없음) | (6) \_\_\_ |

<details>
<summary>🔑 정답 보기 (클릭)</summary>

| 확률 가정 | MLE로 도출되는 활성화 함수 | MLE로 도출되는 손실 함수 |
|---------|:-------------------:|:-------------------:|
| 정규분포 | (해당 없음) | **(1) MSE / RMSE** |
| 베르누이 분포 | **(2) 시그모이드** | **(3) 이진 교차 엔트로피 (BCE)** |
| 카테고리 분포 | **(4) 소프트맥스** | **(5) 범주형 교차 엔트로피 (CCE)** |
| 라플라스 분포 | (해당 없음) | **(6) MAE** |

모두 동일한 원리: **"데이터가 어떤 확률 분포를 따른다고 가정 → MLE → 함수/손실이 필연적으로 도출"**
</details>

---
## 🏁 마무리

이 보충 노트북에서 다룬 내용을 정리하면 다음과 같다.

| 보충 주제 | 핵심 한 줄 요약 |
|---------|-------------|
| 중심극한정리 실험 | 요인 수를 늘려가면 종 모양이 서서히 나타나는 것을 직접 확인했다. |
| $2\sigma^2$ 생략 이유 | 모든 샘플에 공통인 상수이므로 최적화 결과에 영향을 주지 않아 생략한 것이다. |
| 라플라스 분포 → MAE | 정규분포에서 MSE가 나오듯, 라플라스 분포에서 MAE가 도출된다. 꼬리가 두꺼워 이상치에 강하다. |
| 승산비(Odds) | "3대 1로 이긴다"의 그 비율이다. 로그를 씌우면 $-\infty \sim +\infty$ 범위가 되어 선형 회귀와 연결된다. |
| 이진 교차 엔트로피 | 베르누이 분포 + MLE에서 필연적으로 도출되는 이진 분류의 손실 함수이다. |
| 수치 안정성 트릭 | 최댓값을 빼도 결과는 동일하지만 `exp` 오버플로우를 방지할 수 있다. |
| 범주형 교차 엔트로피 | 카테고리 분포 + MLE에서 도출되며, BCE의 일반화이다. |
| 소프트맥스 → 시그모이드 | 클래스를 2개로 줄이면 소프트맥스 공식이 시그모이드와 수학적으로 동일해진다. |

> 모든 공식의 뿌리는 하나다: **"데이터의 확률 분포를 가정하고, MLE로 풀어라."** 🌱